In [5]:
import numpy as np
from typing import List, Dict, Any
from bertopic import BERTopic
import bertopic
from hdbscan import HDBSCAN
from umap import UMAP
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
import traceback
import pandas as pd

import sys
import os
# Adds the parent directory ('..') to the Python path
sys.path.append(os.path.abspath(os.path.join("..")))
from app.models.topic_result import TopicResult


In [6]:
def get_default_params() -> Dict[str, Any]:
    return {
        'bertopic_min_topic_size': 15,
        'bertopic_embedding_model': 'all-MiniLM-L6-v2',
        'bertopic_calculate_probabilities': True,
        'umap_n_neighbors': 15,
        'umap_n_components': 2,
        'umap_metric': 'cosine',
        'umap_low_memory': False,
        'hdbscan_min_samples': 10,
        'hdbscan_metric': 'euclidean'
    }

In [7]:
def train(texts: List[str], **params) -> TopicResult:
    """
    Train BERTopic model and return a standardized TopicResult.
    """
    try:
        # 1. Configure UMAP
        umap_model = UMAP(
            n_neighbors=int(params.get('umap_n_neighbors', 15)),
            n_components=int(params.get('umap_n_components', 5)),
            metric=params.get('umap_metric', 'cosine'),
            low_memory=bool(params.get('umap_low_memory', False)),
            random_state=8
        )

        # 2. Configure HDBSCAN
        clusterer = HDBSCAN(
            min_cluster_size=int(params.get('bertopic_min_topic_size', 5)),
            min_samples=int(params.get('hdbscan_min_samples', 5)),
            metric=params.get('hdbscan_metric', 'euclidean'),
            prediction_data=True
        )

        # 3. Configure Vectorizer and CT-IDF
        vectorizer = CountVectorizer(stop_words="english")
        ctfidf_model = ClassTfidfTransformer(bm25_weighting=True, reduce_frequent_words=True)

        # 4. Initialize BERTopic
        model = BERTopic(
            min_topic_size=int(params.get('bertopic_min_topic_size', 10)),
            embedding_model=params.get('bertopic_embedding_model', 'all-MiniLM-L6-v2'),
            calculate_probabilities=bool(params.get('bertopic_calculate_probabilities', True)),
            umap_model=umap_model,
            hdbscan_model=clusterer,
            vectorizer_model=vectorizer,
            ctfidf_model=ctfidf_model
        )

        # Fit the model
        topics, probs = model.fit_transform(texts)

        # 1. Extract Topics (term -> weight)
        # BERTopic stores topics as (word, weight) tuples in get_topic()
        topics_list = []
        # Topic -1 is outliers, we usually exclude it from the top topics list
        for topic_id in sorted(model.get_topics().keys()):
            if topic_id == -1:
                continue
            topic_data = model.get_topic(topic_id)
            # Convert [(word, weight), ...] to {word: weight, ...}
            topics_list.append({word: weight for word, weight in topic_data})

        # 2. Extract Doc-Topic assignments
        # If calculate_probabilities=True, probs is a numpy array [n_docs, n_topics]
        doc_topics = []
        if isinstance(probs, np.ndarray):
            # Ensure probs is 2D even if only one topic was found
            if probs.ndim == 1:
                probs = probs.reshape(-1, 1)

            # Map each doc to a dict of {topic_id: probability}
            # BERTopic's probs array corresponds to the topics in model.get_topics()
            # Topic -1 is not usually in the probs array
            topic_names = list(model.get_topics().keys())
            # Filter out -1 for probability mapping
            filtered_topics = [t for t in topic_names if t != -1]

            for doc_prob in probs:
                # Map prob index to actual topic_id
                doc_map = {filtered_topics[i]: prob for i, prob in enumerate(doc_prob)}
                doc_topics.append(doc_map)
        else:
            # Fallback to hard assignment
            for t in topics:
                doc_topics.append({t: 1.0})

        # 3. Metadata & Model Info
        metadata = {
            'model': model,
            'texts': texts,
            'num_topics': len(model.get_topics()) - 1 # exclude -1
        }

        model_info = {
            'algorithm': 'BERTopic',
            'version': bertopic.__version__,
            'params': str(params),
            'num_documents': str(len(texts)),
            'num_topics': str(len(model.get_topics()) - 1)
        }

        return TopicResult(
            topics=topics_list,
            doc_topics=doc_topics,
            metadata=metadata,
            model_info=model_info
        )

    except Exception as e:
        call_stack = traceback.format_exc()
        raise RuntimeError(f"BERTopic training failed: {str(e)}\n\n {call_stack}") from e

In [9]:
df = pd.read_csv("../sample_dataset.csv")
params = get_default_params()
result = train(texts=df["Abstract"], **params)

In [10]:
topic_model = result.metadata["model"]

In [11]:
print(result.metadata["num_topics"])

5


In [12]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,86,-1_smart_data_algorithm_city,"[smart, data, algorithm, city, cities, securit...",[Objective An information security evaluation ...
1,0,539,0_ai_public_intelligence_artificial,"[ai, public, intelligence, artificial, governm...",[Purpose: The purpose of this paper is to expl...
2,1,133,1_smart_traffic_learning_cities,"[smart, traffic, learning, cities, city, detec...",[Background. The environment has been signific...
3,2,25,2_covid_19_health_epidemic,"[covid, 19, health, epidemic, pandemic, chest,...",[The worldwide pandemic Coronavirus (Covid-19)...
4,3,19,3_neural_bp_risk_allocation,"[neural, bp, risk, allocation, network, perfor...",[This study investigates the prioritization an...
5,4,19,4_algorithm_problem_genetic_particle,"[algorithm, problem, genetic, particle, schedu...",[The environment in earthquake-stricken areas ...


In [14]:
intertopci_viz = topic_model.visualize_topics()

In [18]:
intertopci_viz.to_dict()

{'data': [{'customdata': array([[0, 'ai | public | intelligence | artificial | government', 539],
          [1, 'smart | traffic | learning | cities | city', 133],
          [2, 'covid | 19 | health | epidemic | pandemic', 25],
          [3, 'neural | bp | risk | allocation | network', 19],
          [4, 'algorithm | problem | genetic | particle | scheduling', 19]],
         dtype=object),
   'hovertemplate': '<b>Topic %{customdata[0]}</b><br>%{customdata[1]}<br>Size: %{customdata[2]}',
   'legendgroup': '',
   'marker': {'color': '#B0BEC5',
    'size': array([539, 133,  25,  19,  19]),
    'sizemode': 'area',
    'sizeref': 0.336875,
    'symbol': 'circle',
    'line': {'color': 'DarkSlateGrey', 'width': 2}},
   'mode': 'markers',
   'name': '',
   'orientation': 'v',
   'showlegend': False,
   'x': array([-10.356627, -10.743371,  -9.601367, -10.403447, -10.253813],
         dtype=float32),
   'xaxis': 'x',
   'y': array([14.741306, 16.38574 , 15.514859, 15.536822, 16.893059],
       

In [19]:
heatmap_viz = topic_model.visualize_heatmap()

In [21]:
heatmap_viz.to_dict()

{'data': [{'coloraxis': 'coloraxis',
   'name': '0',
   'x': ['0_ai_public_intelligence',
    '1_smart_traffic_learning',
    '2_covid_19_health',
    '3_neural_bp_risk',
    '4_algorithm_problem_genetic'],
   'y': ['0_ai_public_intelligence',
    '1_smart_traffic_learning',
    '2_covid_19_health',
    '3_neural_bp_risk',
    '4_algorithm_problem_genetic'],
   'z': array([[0.9999999 , 0.4274909 , 0.5480826 , 0.57863474, 0.30286103],
          [0.4274909 , 1.0000004 , 0.56277156, 0.56603706, 0.5208762 ],
          [0.5480826 , 0.56277156, 1.0000001 , 0.57009065, 0.4521345 ],
          [0.57863474, 0.56603706, 0.57009065, 1.0000001 , 0.4688803 ],
          [0.30286103, 0.5208762 , 0.4521345 , 0.4688803 , 1.        ]],
         dtype=float32),
   'type': 'heatmap',
   'xaxis': 'x',
   'yaxis': 'y',
   'hovertemplate': 'x: %{x}<br>y: %{y}<br>Similarity Score: %{z}<extra></extra>'}],
 'layout': {'template': {'data': {'histogram2dcontour': [{'type': 'histogram2dcontour',
      'colorbar': {

In [22]:
barchart_viz = topic_model.visualize_barchart()

In [24]:
barchart_viz.to_dict()

{'data': [{'marker': {'color': '#D55E00'},
   'orientation': 'h',
   'x': [0.30521055366908506,
    0.3113423457745851,
    0.3128950191787837,
    0.3382045695282492,
    0.3623270285805497],
   'y': ['government  ', 'artificial  ', 'intelligence  ', 'public  ', 'ai  '],
   'type': 'bar',
   'xaxis': 'x',
   'yaxis': 'y'},
  {'marker': {'color': '#0072B2'},
   'orientation': 'h',
   'x': [0.35542894326639424,
    0.3557287312352084,
    0.37481403198446306,
    0.3969470548479714,
    0.40417279761085256],
   'y': ['city  ', 'cities  ', 'learning  ', 'traffic  ', 'smart  '],
   'type': 'bar',
   'xaxis': 'x2',
   'yaxis': 'y2'},
  {'marker': {'color': '#CC79A7'},
   'orientation': 'h',
   'x': [0.38027515452117955,
    0.40478459976263453,
    0.5097443140320257,
    0.5782343708554053,
    0.5820878579508149],
   'y': ['pandemic  ', 'epidemic  ', 'health  ', '19  ', 'covid  '],
   'type': 'bar',
   'xaxis': 'x3',
   'yaxis': 'y3'},
  {'marker': {'color': '#E69F00'},
   'orientation':

In [25]:
term_viz = topic_model.visualize_term_rank()

In [27]:
term_viz.to_dict()

{'data': [{'hovertext': '<b>Topic -1</b>:smart_data_algorithm_city_cities_s',
   'line': {'color': 'black', 'width': 1.5},
   'mode': 'lines+lines',
   'name': '',
   'opacity': 0.1,
   'x': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10]),
   'y': array([0.36006661, 0.33244506, 0.32678936, 0.32622972, 0.32206243,
          0.3176733 , 0.31037787, 0.29585704, 0.28585451, 0.27937652]),
   'type': 'scatter'},
  {'hovertext': '<b>Topic 0</b>:ai_public_intelligence_artificial_g',
   'line': {'color': 'black', 'width': 1.5},
   'mode': 'lines+lines',
   'name': '',
   'opacity': 0.1,
   'x': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10]),
   'y': array([0.36232703, 0.33820457, 0.31289502, 0.31134235, 0.30521055,
          0.28525094, 0.28031446, 0.27415392, 0.26995579, 0.26995081]),
   'type': 'scatter'},
  {'hovertext': '<b>Topic 1</b>:smart_traffic_learning_cities_city_',
   'line': {'color': 'black', 'width': 1.5},
   'mode': 'lines+lines',
   'name': '',
   'opacity': 0.1,
   'x': array

In [28]:
hierarcy_viz = topic_model.visualize_hierarchy()

In [31]:
hierarcy_viz.to_dict()

{'data': [{'hoverinfo': 'text',
   'marker': {'color': 'rgb(61,153,112)'},
   'mode': 'lines',
   'x': array([0.        , 0.44642621, 0.44642621, 0.        ]),
   'xaxis': 'x',
   'y': array([-25., -25., -35., -35.]),
   'yaxis': 'y',
   'type': 'scatter'},
  {'hoverinfo': 'text',
   'marker': {'color': 'rgb(61,153,112)'},
   'mode': 'lines',
   'x': array([0.        , 0.60599617, 0.60599617, 0.44642621]),
   'xaxis': 'x',
   'y': array([-15., -15., -30., -30.]),
   'yaxis': 'y',
   'type': 'scatter'},
  {'hoverinfo': 'text',
   'marker': {'color': 'rgb(61,153,112)'},
   'mode': 'lines',
   'x': array([0.60599617, 0.63921047, 0.63921047, 0.        ]),
   'xaxis': 'x',
   'y': array([-22.5, -22.5, -45. , -45. ]),
   'yaxis': 'y',
   'type': 'scatter'},
  {'hoverinfo': 'text',
   'marker': {'color': 'rgb(61,153,112)'},
   'mode': 'lines',
   'x': array([0.        , 0.70977448, 0.70977448, 0.63921047]),
   'xaxis': 'x',
   'y': array([ -5.  ,  -5.  , -33.75, -33.75]),
   'yaxis': 'y',
   